In [ ]:
# ========== 导入 + 环境：微调定价器（Fine Tuning Pricer）前置准备 ==========

# 标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# 标准库 re：本练习里可能用于从模型回复里抽价格数字（后续单元也可能用到）
import re
# 标准库 json：把 messages 列表序列化成 JSONL 行、以及 pretty-print 示例
import json
# 标准库 time：上传失败时 sleep 退避重试
import time
# pathlib.Path：用面向对象方式拼 jsonl 路径、建目录
from pathlib import Path
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# huggingface_hub.login：用 HF_TOKEN 登录 Hugging Face Hub（拉数据集要用）
from huggingface_hub import login
# OpenAI 客户端：上传 fine-tune 文件、创建/轮询微调任务、推理
from openai import OpenAI
# numpy：数值计算（本格主要随 sklearn 指标一起导入）
import numpy as np
# sklearn 回归指标：MAE / MSE / R²（evaluate 内部或后续对比可能用到）
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 课程自定义：Item 商品样本（含 summary / price），可从 Hub 拉取
from pricer.items import Item
# 课程自定义：evaluate 批量评测 + Tester 可视化评测工具
from pricer.evaluator import evaluate, Tester

# 加载 .env；override=True 表示已有环境变量也会被 .env 覆盖
load_dotenv(override=True)
# 读取 Hugging Face token（没有则为 None，后面 if 跳过登录）
hf_token = os.environ.get("HF_TOKEN")
# 有 token 才登录；add_to_git_credential=True 顺带写进 git 凭证缓存
if hf_token:
    login(hf_token, add_to_git_credential=True)

# 创建 OpenAI 客户端：默认从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()


In [ ]:
# ========== 从 Hugging Face Hub 拉取商品定价数据集 ==========

# LITE_MODE=True：用小样本 items_lite 快速试跑；False 则换 items_full
LITE_MODE = True
# Hub 上数据集所属用户名（课程作者 ed-donner）
USERNAME = "ed-donner"
# 按开关拼出数据集全名：用户名/仓库名
dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"

# Item.from_hub：一次返回 train / val / test 三个列表
train, val, test = Item.from_hub(dataset_name)
# 打印各分割规模（千分位逗号），确认数据拉下来了
print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")
# 抽样看第一条：标题截断 50 字 + 真实价格，建立「输入文本 → 标签价格」直觉
print(f"Example: {train[0].title[:50]}... → ${train[0].price:.2f}")


In [ ]:
# ========== 构造 Chat Fine-Tuning 的 messages（训练 vs 推理） ==========

# system prompt：约束模型「只回 $X.XX 价格、不要解释」——字符串影响行为，保持英文原样
SYSTEM_PROMPT = (
    "You are a product pricing expert. "
    "Given a product description, estimate its fair market price in USD. "
    "Reply with only the price in the format $X.XX (e.g. $29.99). No explanation."
)

def messages_for(item: Item):
    """训练/验证用：system + user + assistant（assistant 里放真值价格，供监督微调）。"""
    # user 内容：任务说明 + 商品摘要 item.summary（prompt 字符串保持英文）
    user_content = f"Estimate the price of this product.\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        # assistant 完成：真值价格格式化为 $xx.xx —— 这是 supervised fine-tune 的标签
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def test_messages_for(item: Item):
    """推理用：只有 system + user，不给 assistant，让模型自己补全价格。"""
    user_content = f"Estimate the price of this product.\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

# 打印第一条训练 messages 的 JSON 预览（截断 600 字符），确认结构正确
print("Example training messages:")
print(json.dumps(messages_for(train[0]), indent=2)[:600], "...")


In [ ]:
# ========== 截取微调子集：控制成本与耗时 ==========

# 训练集条数上限（OpenAI fine-tune 按 token/次数计费，先用小集合验证流程）
N_TRAIN = 300
# 验证集条数：训练过程中监控过拟合
N_VAL = 80

# 从完整 train/val 切前 N 条作为本次微调数据
fine_tune_train = train[:N_TRAIN]
fine_tune_validation = val[:N_VAL]
# 确认切片长度符合预期
print(f"Fine-tune train: {len(fine_tune_train)}  |  Fine-tune val: {len(fine_tune_validation)}")


In [ ]:
# ========== 写成 JSONL + 上传到 OpenAI Files（带 500 重试） ==========

def make_jsonl(items):
    """把每条 Item 变成一行 JSON：{"messages": [...]}，多行拼成 JSONL 文本。"""
    lines = []
    for item in items:
        # 训练格式：含 assistant 真值
        messages = messages_for(item)
        # 一行一个 JSON 对象（OpenAI fine-tune 要求 JSONL）
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)

def write_jsonl(items, path: str):
    """确保父目录存在后，把 JSONL 写到磁盘。"""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))

# 本地 jsonl 目录与两个文件路径
JSONL_DIR = Path("jsonl")
train_path = JSONL_DIR / "fine_tune_train.jsonl"
val_path = JSONL_DIR / "fine_tune_validation.jsonl"
# 写出训练 / 验证 JSONL
write_jsonl(fine_tune_train, train_path)
write_jsonl(fine_tune_validation, val_path)
print(f"Wrote {train_path} ({len(fine_tune_train)} examples), {val_path} ({len(fine_tune_validation)} examples)")

def upload_with_retry(file_path, purpose="fine-tune", max_retries=3):
    """上传文件；遇到 InternalServerError(500) 则指数式拉长等待后重试。"""
    # 延迟导入：只在上传路径需要时引用异常类型
    from openai import InternalServerError
    for attempt in range(1, max_retries + 1):
        try:
            # 二进制打开后交给 OpenAI Files API；purpose 必须是 fine-tune
            with open(file_path, "rb") as f:
                return openai.files.create(file=f, purpose=purpose)
        except InternalServerError as e:
            # 最后一次仍失败就抛出
            if attempt == max_retries:
                raise
            # 退避：第 1 次等 60s，第 2 次 120s…
            wait = 60 * attempt
            print(f"500 on upload (attempt {attempt}/{max_retries}). Retrying in {wait}s...")
            time.sleep(wait)
    return None

# 若客户端不可用则跳过上传（防御分支；通常 OpenAI() 总会建出对象）
if not openai:
    train_file = None
    validation_file = None
    print("Skipped (no OPENAI_API_KEY).")
else:
    # 分别上传训练与验证文件，拿到 file.id 供创建 job 使用
    train_file = upload_with_retry(train_path)
    validation_file = upload_with_retry(val_path)
    print(f"Uploaded train file: {train_file.id}, validation file: {validation_file.id}")


In [ ]:
# ========== 选择超参实验并创建 Fine-Tuning Job ==========

# 基座模型 id：必须与 OpenAI 当前支持的 fine-tune 模型名一致（勿擅自改写）
BASE_MODEL = "gpt-4.1-nano-2025-04-14"


# 多组超参字典：方便对比 epochs / batch_size / lr multiplier
EXPERIMENTS = {
    "baseline": {"n_epochs": 1, "batch_size": 1},
    "more_epochs": {"n_epochs": 2, "batch_size": 1},
    "larger_batch": {"n_epochs": 1, "batch_size": 2},
    "conservative": {"n_epochs": 2, "batch_size": 1, "learning_rate_multiplier": 0.5},
}

# 本次选用的实验名；改这里即可切换超参，不必改 create 调用
CHOSEN = "more_epochs"
# 取出对应超参字典
hyperparams = EXPERIMENTS[CHOSEN]
print(f"Using experiment: {CHOSEN}  →  {hyperparams}")

# 没有客户端或没上传成功 → 跳过创建任务
if not openai or not train_file:
    job_id = None
    print("Skipped (no OpenAI client or uploaded files).")
else:
    # 创建微调任务：绑定训练/验证文件、基座模型、随机种子、超参、后缀名
    job = openai.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters=hyperparams,
        # suffix 会出现在最终 fine_tuned_model 名字里，便于辨认实验
        suffix=f"pricer_{CHOSEN}",
    )
    # 记下 job.id，下一格轮询状态要用
    job_id = job.id
    print(f"Fine-tuning job created: {job_id}")


In [ ]:
# ========== 轮询微调任务直到成功 / 失败 ==========

# 若上一格没跑完，globals 里可能没有 job_id；用 get 安全读取
job_id = globals().get("job_id", None)
# 成功后会填入最终模型名（ft:...）
fine_tuned_model_name = None
if job_id and openai:
    # 忙等循环：每 30 秒查一次状态
    while True:
        job = openai.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        # 同行打印状态，形成进度感
        print(status, end=" ")
        if status == "succeeded":
            # 成功：取出 fine_tuned_model 供推理
            fine_tuned_model_name = job.fine_tuned_model
            print(f"\nDone. Model: {fine_tuned_model_name}")
            break
        if status == "failed":
            # 失败：打印 error 字段并抛异常中断
            print(f"\nJob failed: {getattr(job, 'error', None)}")
            raise RuntimeError("Fine-tuning job failed")
        # 仍在排队/训练中：睡 30 秒再查
        time.sleep(30)
else:
    print("Skipped (no job_id).")


In [ ]:
# ========== 用微调后模型在 test 集上评测 ==========

# 评测样本量：控制 API 调用次数与等待时间
EVAL_SIZE = 200

def fine_tuned_pricer(item: Item):
    """对单条商品调用微调模型；缺失客户端/模型名时返回占位 "0.00"。"""
    if not openai or not fine_tuned_model_name:
        return "0.00"
    # chat.completions：messages 用推理版（无 assistant 真值）
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        # 价格很短，限制 max_tokens 省钱也减少胡言
        max_tokens=16,
    )
    # 取出文本并 strip；空则变 ""
    return (response.choices[0].message.content or "").strip()

# 把预测函数交给课程 evaluate：在 test 上算误差并出报告
predictor = fine_tuned_pricer
evaluate(predictor, test, size=EVAL_SIZE)
